# Running a Compute Task Without `await`

`ComputeClient` is asynchronous: every call is `await client.<topic>.<task>.run(...)`.
That reads fine at the top of a notebook cell, but it is one more thing to know, and it rules
out a plain `.py` script entirely.

`SyncComputeClient` is the same engine with the `await` removed. Discovery, validation,
reference resolution and result hydration are all unchanged -- the only difference is that a
call blocks until the task finishes instead of handing back a coroutine. Everything below
works the same way in a script.

Which one to use is a choice, not a mode: pick `ComputeClient` inside `async` code, and
`SyncComputeClient` everywhere else.

## Sign in on the same event loop

This is the one thing to get right. A connection is bound to the event loop it was first used
on, so signing in with `await manager.login()` ties the connection to the notebook kernel's
loop, and the blocking client -- which runs on its own -- can never drive it.

`run_sync` runs a coroutine on that loop and waits for it. Using it for the sign-in as well
keeps everything on one loop, and leaves no `await` anywhere in the notebook. If a context
signed in with `await` is used anyway, the first call says so and names the two ways out.

In [ ]:
from evo.notebooks import ServiceManagerWidget

from evo.compute import run_sync

manager = run_sync(
    ServiceManagerWidget.with_auth_code(client_id="your-client-id", cache_location="./notebook-data").login()
)

## Open the catalogue

The client is instance-bound and takes the same arguments as `ComputeClient`. It is not a
context manager; there is nothing to enter or close.

In [ ]:
from evo.compute import SyncComputeClient

client = SyncComputeClient(manager)

## Find a task

Discovery is deliberately lazy: reaching `client.geostatistics.kriging` costs nothing, and
the catalogue is only fetched the first time a task is actually run. So `dir()` and tab
completion are empty on a fresh client -- they read the cache, and there is nothing in it
yet.

To see what your organization can run, ask the discovery client. That is the authoritative
list because it fetches, and once it has, the namespace completes from then on -- until the
cache expires, five minutes later by default.

In [ ]:
from evo.compute import DiscoveryClient

discovery = DiscoveryClient.from_context(manager)
for task in sorted(run_sync(discovery.list_tasks()), key=lambda task: (task.topic, task.name)):
    print(f"{task.topic}.{task.name}", f"v{task.version}" if task.version else "")

## Load the objects to run it on

`run_sync` is not only for signing in: any coroutine from the rest of the SDK can go through
it, and because it is the same loop the client uses, the objects it returns are usable as
task parameters.

In [ ]:
from evo.objects.typed import object_from_uuid

# Replace these with objects from your own workspace.
samples = run_sync(object_from_uuid(manager, "00000000-0000-0000-0000-000000000001"))
variogram = run_sync(object_from_uuid(manager, "00000000-0000-0000-0000-000000000002"))
target_model = run_sync(object_from_uuid(manager, "00000000-0000-0000-0000-000000000003"))

# The attribute to estimate from has to be one of these. `%load_ext evo.widgets` renders
# them as a table instead, but printing the names works in a plain script too.
print([attribute.name for attribute in samples.attributes])

## Run it

The parameters are the ones the task publishes, and the objects and attributes you already
hold can be passed directly -- the engine turns them into the references the schema asks for.
A missing or misspelled parameter is rejected before anything is submitted.

Prefer the typed attribute (`samples.attributes["..."]`) over a bare name. Both are accepted,
but a name is turned into a JMESPath expression matched against the object, so a name that is
not there selects nothing and the task fails on the service rather than here. An attribute you
hold carries its own key, so there is nothing to guess. Indexing a name the object does *not*
have -- `grid.attributes["kriged_grade"]` -- is how you say "create this one".

In [ ]:
# Search the variogram's own ellipsoid, widened. Guessed ranges find no samples at all.
search_ellipsoid = variogram.get_ellipsoid().scaled(2.0)

result = client.geostatistics.kriging.run(
    source=samples.attributes["grade"],
    target=target_model.attributes["kriged_grade"],
    kriging_method={"type": "ordinary"},
    variogram=variogram,
    neighborhood={"ellipsoid": search_ellipsoid.to_dict(), "max_samples": 20},
    block_discretisation={"nx": 3, "ny": 3, "nz": 2},
)

## Read the results

The result is the payload the platform sent, so it still indexes and prints as a dictionary.
On top of that, the parts the task marked as outputs load the objects and attributes they
refer to -- and on this client those loaders block too, so there is still no `await`.

In [ ]:
print(result["target"]["reference"])
print(result.target.attribute.name)

In [ ]:
estimated = result.target.load()
estimated

In [ ]:
result.target.attribute.to_dataframe().head()

## The same run, asynchronously

For comparison, and for when you are already inside `async` code. The two clients are
siblings over the same engine, so the job submitted is identical -- only the `await` differs.

In [ ]:
from evo.compute import ComputeClient


async def krige(manager):
    client = ComputeClient(manager)
    result = await client.geostatistics.kriging.run(
        source=samples.attributes["grade"],
        target=target_model.attributes["kriged_grade"],
        kriging_method={"type": "ordinary"},
        variogram=variogram,
        neighborhood={"ellipsoid": search_ellipsoid.to_dict(), "max_samples": 20},
        block_discretisation={"nx": 3, "ny": 3, "nz": 2},
    )
    return await result.target.load()

## If something goes wrong

- **`SyncBridgeError: this connection was first used on a different event loop`** -- the
  context was signed in with `await manager.login()`. Restart the kernel and sign in through
  `run_sync` as above, or switch to `ComputeClient`.
- **`SyncBridgeError: run_sync() was called from inside the event loop it runs coroutines
  on`** -- a blocking call was made from inside a coroutine that `run_sync` is already
  running. Await the coroutine there instead.
- **`ParameterValidationError`** -- the parameters did not match the task's schema. The
  message names the parameter; nothing was submitted.